In [1]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import re
import time
from html import unescape
from collections import Counter

In [2]:
CATEGORY_MAP = {
    "economics": [
        "econ.EM",   # Econometrics
        "econ.GN",   # General Economics
        "econ.TH"    # Theoretical Economics
    ],
    "electrical_engineering": [
        "eess.AS",   # Audio and Speech Processing
        "eess.IV",   # Image and Video Processing
        "eess.SP",   # Signal Processing
        "eess.SY"    # Systems and Control
    ],
    "biology": [
        "q-bio.BM",  # Biomolecules
        "q-bio.GN",  # Genomics
        "q-bio.NC"   # Neurons and Cognition
    ]
}

In [3]:
def flatten_category_map(category_map):
    rows = []
    for broad_subject, subcats in category_map.items():
        for subcat in subcats:
            rows.append((broad_subject, subcat))
    return rows

flat_categories = flatten_category_map(CATEGORY_MAP)
flat_categories

[('economics', 'econ.EM'),
 ('economics', 'econ.GN'),
 ('economics', 'econ.TH'),
 ('electrical_engineering', 'eess.AS'),
 ('electrical_engineering', 'eess.IV'),
 ('electrical_engineering', 'eess.SP'),
 ('electrical_engineering', 'eess.SY'),
 ('biology', 'q-bio.BM'),
 ('biology', 'q-bio.GN'),
 ('biology', 'q-bio.NC')]

In [ ]:
def fetch_primary_category_papers(category, target_primary_count=2000, batch_size=500, sort_by="submittedDate"):
    all_rows = []
    start = 0

    BASE_URL = "http://export.arxiv.org/api/query"

    ns = {
        "atom": "http://www.w3.org/2005/Atom",
        "arxiv": "http://arxiv.org/schemas/atom"
    }

    stop_reason = "unknown"

    while len(all_rows) < target_primary_count and start < 30000:
        current_batch = min(batch_size, 2000)

        params = {
            "search_query": f"cat:{category}",
            "start": start,
            "max_results": current_batch,
            "sortBy": sort_by,
            "sortOrder": "descending"
        }

        response = requests.get(BASE_URL, params=params, timeout=30)
        response.raise_for_status()

        root = ET.fromstring(response.text)
        entries = root.findall("atom:entry", ns)

        if not entries:
            stop_reason = "no more entries returned"
            break

        kept_this_batch = 0

        for entry in entries:
            title = entry.findtext("atom:title", default="", namespaces=ns)
            summary = entry.findtext("atom:summary", default="", namespaces=ns)
            published = entry.findtext("atom:published", default="", namespaces=ns)
            updated = entry.findtext("atom:updated", default="", namespaces=ns)
            paper_id = entry.findtext("atom:id", default="", namespaces=ns)

            authors = entry.findall("atom:author", ns)
            author_names = [a.findtext("atom:name", default="", namespaces=ns) for a in authors]

            categories_xml = entry.findall("atom:category", ns)
            category_terms = [c.attrib.get("term", "") for c in categories_xml]

            primary_category = ""
            primary_cat_elem = entry.find("arxiv:primary_category", ns)
            if primary_cat_elem is not None:
                primary_category = primary_cat_elem.attrib.get("term", "")

            if primary_category != category:
                continue

            row = {
                "id": paper_id,
                "title": title.strip(),
                "abstract": summary.strip(),
                "published": published,
                "updated": updated,
                "authors": ", ".join(author_names),
                "primary_category": primary_category,
                "all_categories": ", ".join(category_terms),
                "queried_category": category
            }

            all_rows.append(row)
            kept_this_batch += 1

            if len(all_rows) >= target_primary_count:
                stop_reason = "target reached"
                break

        print(
            f"{category}: start={start}, batch_returned={len(entries)}, "
            f"kept_primary={kept_this_batch}, total_kept={len(all_rows)}"
        )

        start += current_batch
        time.sleep(3)

    if start >= 30000 and len(all_rows) < target_primary_count:
        stop_reason = "hit start limit"

    print(f"{category}: STOPPED because -> {stop_reason}")
    return pd.DataFrame(all_rows)

In [5]:
TARGET_PER_PRIMARY_CATEGORY = 2000

all_dfs = []

for broad_subject, subcat in flat_categories:
    print(f"\nFetching primary papers for {subcat} under {broad_subject}...")
    
    temp_df = fetch_primary_category_papers(
        category=subcat,
        target_primary_count=TARGET_PER_PRIMARY_CATEGORY,
        batch_size=1000
    )
    
    temp_df["broad_subject"] = broad_subject
    all_dfs.append(temp_df)

df_raw = pd.concat(all_dfs, ignore_index=True)

print("\nTotal rows collected:", len(df_raw))


Fetching primary papers for econ.EM under economics...


econ.EM: start=0, batch_returned=1000, kept_primary=695, total_kept=695
econ.EM: start=1000, batch_returned=1000, kept_primary=727, total_kept=1422
econ.EM: start=2000, batch_returned=1000, kept_primary=578, total_kept=2000
econ.EM: STOPPED because -> target reached

Fetching primary papers for econ.GN under economics...
econ.GN: start=0, batch_returned=1000, kept_primary=747, total_kept=747
econ.GN: start=1000, batch_returned=1000, kept_primary=772, total_kept=1519
econ.GN: start=2000, batch_returned=1000, kept_primary=481, total_kept=2000
econ.GN: STOPPED because -> target reached

Fetching primary papers for econ.TH under economics...
econ.TH: start=0, batch_returned=1000, kept_primary=745, total_kept=745
econ.TH: start=1000, batch_returned=1000, kept_primary=717, total_kept=1462
econ.TH: start=2000, batch_returned=1000, kept_primary=538, total_kept=2000
econ.TH: STOPPED because -> target reached

Fetching primary papers for eess.AS under electrical_engineering...
eess.AS: start=0, 

In [6]:
print(df_raw["primary_category"].value_counts())

primary_category
econ.EM     2000
econ.GN     2000
econ.TH     2000
eess.AS     2000
eess.IV     2000
eess.SP     2000
eess.SY     2000
q-bio.BM    2000
q-bio.GN    2000
q-bio.NC    2000
Name: count, dtype: int64


In [7]:
df_raw.to_csv("data/arxiv_raw_metadata.csv", index=False)
df_raw.head()

,id,title,abstract,published,updated,authors,primary_category,all_categories,queried_category,broad_subject
0,http://arxiv.org/abs/2603.21699v1,A Job I Like or a Job I Can Get: Designing Job...,Recommendation systems (RSs) are increasingly ...,2026-03-23T08:37:06Z,2026-03-23T08:37:06Z,"Guillaume Bied, Philippe Caillou, Bruno Crépon...",econ.EM,"econ.EM, stat.ML",econ.EM,economics
1,http://arxiv.org/abs/2603.21004v1,Power Bounds and Efficiency Loss for Asymptoti...,We characterize the maximal attainable power-s...,2026-03-22T01:36:14Z,2026-03-22T01:36:14Z,"Marcelo J. Moreira, Geert Ridder, Mahrad Shari...",econ.EM,"econ.EM, math.ST",econ.EM,economics
2,http://arxiv.org/abs/2603.20936v1,Two Approaches to Direct Estimation of Riesz R...,The Riesz representer is a central object in s...,2026-03-21T20:35:13Z,2026-03-21T20:35:13Z,David Bruns-Smith,econ.EM,"econ.EM, stat.ML",econ.EM,economics
3,http://arxiv.org/abs/2603.20809v1,The Structural Bite: A Methodological Framewor...,We study the employment effects of the 22% inc...,2026-03-21T13:11:29Z,2026-03-21T13:11:29Z,Marcos Lacasa-Cazcarra,econ.EM,econ.EM,econ.EM,economics
4,http://arxiv.org/abs/2603.20464v1,Double Machine Learning for Static Panel Data ...,Panel data methods are widely used in empirica...,2026-03-20T19:49:57Z,2026-03-20T19:49:57Z,"Anna Baiardi, Paul S. Clarke, Andrea A. Naghi,...",econ.EM,"econ.EM, stat.ME, stat.ML",econ.EM,economics


In [8]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = unescape(text)
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

def simple_cleaning(df):
    df = df.copy()

    df["title_clean"] = df["title"].apply(clean_text)
    df["abstract_clean"] = df["abstract"].apply(clean_text)
    df["text"] = df["title_clean"] + " " + df["abstract_clean"]

    df["abstract_word_count"] = df["abstract_clean"].apply(lambda x: len(x.split()))
    df["title_word_count"] = df["title_clean"].apply(lambda x: len(x.split()))

    # remove duplicates by id
    df = df.drop_duplicates(subset=["id"]).reset_index(drop=True)

    # remove very short abstracts
    df = df[df["abstract_word_count"] >= 20].reset_index(drop=True)

    return df

df = simple_cleaning(df_raw)
print("Rows b4 cleaning:", len(df_raw))
print("Rows after cleaning:", len(df))

Rows b4 cleaning: 20000
Rows after cleaning: 19988


In [9]:
# --- handle arXiv versions properly ---

df["base_id"] = df["id"].str.extract(r"(\d+\.\d+)")
df["version"] = df["id"].str.extract(r"v(\d+)$").astype(float)

print("Before dedup:", len(df))

df = df.sort_values(by=["base_id", "version"], ascending=[True, False])
df = df.drop_duplicates(subset="base_id", keep="first").reset_index(drop=True)

print("After dedup:", len(df))

Before dedup: 19988
After dedup: 19977


In [10]:
# compute base_id and version, count versions per paper
df['base_id'] = df['id'].str.extract(r'(\d+\.\d+)(?:v\d+)?$')[0]
df['version'] = df['id'].str.extract(r'v(\d+)$')[0].fillna('1').astype(int)

# count how many versions are present per base_id
version_counts = df.groupby('base_id').size().rename('n_versions')
print('Number of distinct base papers:', version_counts.shape[0])
print('How many base papers have multiple versions:', (version_counts>1).sum())
print('\nDistribution of version counts (n_versions: #base papers):')
print(version_counts.value_counts().sort_index())

# show a few examples of papers with multiple versions
multi = version_counts[version_counts>1].index
if len(multi):
    print('\nExample entries for papers with multiple versions:')
    display(df[df['base_id'].isin(multi)].sort_values(['base_id','version']).head(20))
else:
    print('\nNo multi-version papers found in current dataframe.')

# create a dataframe with latest version per base_id
df_latest = (
    df.sort_values(['base_id','version'], ascending=[True,False])
      .drop_duplicates(subset='base_id', keep='first')
      .reset_index(drop=True)
)
print('\nOriginal rows:', len(df))
print('Rows after keeping latest version only:', len(df_latest))

# assign back if you want to use latest-only going forward:
# df = df_latest.copy()


Number of distinct base papers: 19976
How many base papers have multiple versions: 0

Distribution of version counts (n_versions: #base papers):
n_versions
1    19976
Name: count, dtype: int64

No multi-version papers found in current dataframe.

Original rows: 19977
Rows after keeping latest version only: 19977


In [11]:
df.to_csv("data/arxiv_cleaned_metadata.csv", index=False)
df.head()

,id,title,abstract,published,updated,authors,primary_category,all_categories,queried_category,broad_subject,title_clean,abstract_clean,text,abstract_word_count,title_word_count,base_id,version
0,http://arxiv.org/abs/0704.1169v1,Holographic bound and protein linguistics,The holographic bound in physics constrains th...,2007-04-10T12:04:28Z,2007-04-10T12:04:28Z,"Dirson Jian Li, Shengli Zhang",q-bio.GN,"q-bio.GN, hep-th, q-bio.QM",q-bio.GN,biology,holographic bound and protein linguistics,the holographic bound in physics constrains th...,holographic bound and protein linguistics the ...,66,5,0704.1169,1
1,http://arxiv.org/abs/0704.2191v2,Mismatch Repair Error Implies Chargaff's Secon...,Chargaff's second parity rule holds empiricall...,2007-04-17T16:15:37Z,2007-09-20T15:46:34Z,Bo Deng,q-bio.GN,q-bio.GN,q-bio.GN,biology,mismatch repair error implies chargaff's secon...,chargaff's second parity rule holds empiricall...,mismatch repair error implies chargaff's secon...,155,8,0704.2191,2
2,http://arxiv.org/abs/0704.3826v2,Non-coding DNA programs express adaptation and...,Significant fraction (98.5% in humans) of most...,2007-04-29T05:32:50Z,2007-08-02T16:23:54Z,Mark Ya. Azbel,q-bio.GN,"q-bio.GN, cond-mat.other, nlin.AO, q-bio.OT, q...",q-bio.GN,biology,non-coding dna programs express adaptation and...,significant fraction (98.5% in humans) of most...,non-coding dna programs express adaptation and...,161,9,0704.3826,2
3,http://arxiv.org/abs/0705.1656v1,Chromatin Folding in Relation to Human Genome ...,Three-dimensional (3D) chromatin structure is ...,2007-05-11T14:12:05Z,2007-05-11T14:12:05Z,"Julio Mateos-Langerak, Osdilly Giromus, Wim de...",q-bio.GN,q-bio.GN,q-bio.GN,biology,chromatin folding in relation to human genome ...,three-dimensional (3d) chromatin structure is ...,chromatin folding in relation to human genome ...,226,8,0705.1656,1
4,http://arxiv.org/abs/0705.2143v1,Inverted and mirror repeats in model nucleotid...,We analytically and numerically study the prob...,2007-05-15T13:37:17Z,2007-05-15T13:37:17Z,"Fabrizio Lillo, Marco Spanó",q-bio.GN,"q-bio.GN, q-bio.QM",q-bio.GN,biology,inverted and mirror repeats in model nucleotid...,we analytically and numerically study the prob...,inverted and mirror repeats in model nucleotid...,79,8,0705.2143,1


In [12]:
print(df["abstract_word_count"].describe())

count    19977.000000
mean       165.840717
std         50.673028
min         20.000000
25%        130.000000
50%        162.000000
75%        200.000000
max        539.000000
Name: abstract_word_count, dtype: float64


In [13]:
df[["title", "primary_category", "published"]].sample(10, random_state=42)

,title,primary_category,published
5601,Emergence and maintenance of modularity in neu...,q-bio.NC,2024-05-28T21:07:53Z
8034,Dynamic Structures of Knowledge Production: Ci...,econ.GN,2025-02-02T13:36:25Z
3393,Policy Learning under Biased Sample Selection,econ.EM,2023-04-23T19:54:17Z
4104,Neuroformer: Multimodal and Multitask Generati...,q-bio.NC,2023-10-31T20:17:32Z
16575,Scaling Charitable Incentives: Policy Selectio...,econ.GN,2025-12-31T13:22:30Z
11793,TuningIQA: Fine-Grained Blind Image Quality As...,eess.IV,2025-08-25T12:26:12Z
15614,Optimal Platoon Formation and Stable Benefit A...,eess.SY,2025-12-06T04:23:27Z
15821,Traffic Equilibrium in Mixed-Autonomy Network ...,eess.SY,2025-12-11T01:24:15Z
5107,Expanding Chemical Representation with k-mers ...,q-bio.BM,2024-03-28T21:36:07Z
18912,Shift-Share Designs in Political Science,econ.EM,2026-02-24T00:55:41Z


In [14]:
df["combined_text"] = df["title"] + " " + df["abstract"]

texts = df["combined_text"].tolist()
labels_text = df["primary_category"].tolist()

# Convert labels to numbers
unique_labels = sorted(list(set(labels_text)))
label_map = {label: i for i, label in enumerate(unique_labels)}

labels = [label_map[label] for label in labels_text]

from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

encodings = tokenizer(
    texts,
    padding=True,
    truncation=True,
    max_length=256,  # 512 is slow → 256 is enough
    return_tensors="pt"
)

import torch
from torch.utils.data import Dataset

class TextDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)
    
from torch.utils.data import random_split

dataset = TextDataset(encodings, labels)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8)

from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(unique_labels)
)

from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

model.train()

epochs = 3  # keep small for now

for epoch in range(epochs):
    total_loss = 0
    
    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels_batch = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels_batch
        )

        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1} Loss:", total_loss)

    model.eval()

correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels_batch = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        preds = torch.argmax(outputs.logits, dim=1)

        correct += (preds == labels_batch).sum().item()
        total += labels_batch.size(0)

print("Test Accuracy:", correct / total)

c:\Users\ASUS\anaconda3\envs\is217_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\ASUS\anaconda3\envs\is217_env\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate deve

KeyboardInterrupt: 